# Seleção dos indicadores

* ALVO = fpd

In [1]:
#Bibliotecas
######################
#Manipulação de dados
######################
import pandas as pd
import numpy as np
######################
#Análise estatística
######################
from scipy import stats #Estatística KS
from sklearn.feature_selection import mutual_info_classif #Mutual Information
from sklearn.feature_selection import SelectKBest, f_classif #Fisher Score
##Eliminar os warnings
import warnings
warnings.filterwarnings("ignore")
##Ver todas as colunas do data frame
pd.set_option('display.max_columns', None)

## Fórmulas de suporte

**KS2**

In [2]:
def calcular_ks_2samp(df, alvo, escore):
    # Separa os escores para o grupo 'bons' e remove os valores ausentes
    bons_escore = df.loc[df[alvo] == 0, escore].dropna().rename('bons')
    
    # Separa os escores para o grupo 'maus' e remove os valores ausentes
    maus_escore = df.loc[df[alvo] == 1, escore].dropna().rename('maus')
    
    # Verifica se os grupos não estão vazios antes de calcular
    if bons_escore.empty or maus_escore.empty:
        print("Um dos grupos está vazio após remover valores ausentes.")
        return 0  # Retorna 0 ou outro valor indicando que não foi possível calcular

    # Calcula e retorna a estatística KS
    return stats.ks_2samp(bons_escore, maus_escore).statistic

# 1 - Visão geral dos dados

## Base da QUOD de indicadores

In [3]:
#Ver a codificação do arquivo

#Tentar as codificações mais comuns para arquivos em português
codificacoes = ['latin-1', 'iso-8859-1', 'cp1252', 'utf-8']

for encoding in codificacoes:
    try:
        df = pd.read_csv('base_quod_enriquecida.csv', sep=';', encoding=encoding)
        print(f"✅ Arquivo lido com sucesso usando codificação: {encoding}")
        print(f"📊 Shape do DataFrame: {df.shape}")
        break
    except UnicodeDecodeError:
        print(f"❌ Falha com {encoding}")
        continue
    except Exception as e:
        print(f"⚠️  Outro erro com {encoding}: {e}")
        continue
else:
    print("❌ Nenhuma codificação funcionou")

✅ Arquivo lido com sucesso usando codificação: latin-1
📊 Shape do DataFrame: (169381, 1)


In [4]:
df = pd.read_csv('base_quod_enriquecida.csv', dtype={'cpf': object, 'num_adm': object}, sep = ',', encoding='latin-1')
#Visualização
df.head(3)

,cpf,num_adm,ref_date,safra,data_instalacao,data_cancelamento,dias_empresa,status_aplicacao,dentro_politica,entrada_tv,canal_aquisicao,flag_canal_fisico,quod_score,fpd,flag_churn,churn_curto_prazo,motivo,segmento_quod,segmento_quod_1,segmento_quod_2,segmento_quod_3,segmento_quod_4,escore_faixa,quod_total_restritivos,quod_valor_restritivos,quod_flag_restritivo,GC_AV_GRV01_IF_PF_V5,GC_AV_GRV01_PS_PF_V5,GC_AV_GRV01_TL_PF_V5,GC_EM_ITS01_IF_PF_V5,GC_PD_REC01_IF_PF_V5,GC_PD_REC01_PS_PF_V5,GC_PD_REC01_TL_PF_V5,PF_CP_CNT01_IF_PF_V5,PF_CP_CNT01_PS_PF_V5,PF_CP_CNT01_TL_PF_V5,PF_CP_CNT02_IF_PF_V5,PF_CP_CNT02_PS_PF_V5,PF_CP_CNT02_TL_PF_V5,PF_CP_VOL01_IF_PF_V5,PF_CP_VOL01_PS_PF_V5,PF_CP_VOL01_TL_PF_V5,PF_CP_VOL02_IF_PF_V5,PF_CP_VOL02_PS_PF_V5,PF_CP_VOL02_TL_PF_V5,PF_CP_VOL03_IF_PF_V5,PF_CP_VOL04_IF_PF_V5,PF_EM_HBT01_IF_PF_V5,PF_EM_HBT02_IF_PF_V5,PF_EM_ITS01_IF_PF_V5,PF_EM_ITS02_IF_PF_V5,PF_PC_CPR01_IF_PF_V5,PF_PC_CPR01_PS_PF_V5,PF_PC_CPR01_TL_PF_V5,PF_PC_CPR02_IF_PF_V5,PF_PC_CPR02_PS_PF_V5,PF_PC_CPR02_TL_PF_V5,PF_PC_PRD01_IF_PF_V5,PF_PC_PRD02_IF_PF_V5,PF_PC_TPR01_IF_PF_V5,PF_PC_TPR02_IF_PF_V5,PF_PC_VLR01_IF_PF_V5,PP_AV_AMP01_IF_PF_V5,PP_AV_AMP01_PS_PF_V5,PP_AV_AMP01_TL_PF_V5,PP_AV_AMP02_IF_PF_V5,PP_AV_AMP02_PS_PF_V5,PP_AV_AMP02_TL_PF_V5,PP_AV_AMP03_IF_PF_V5,PP_AV_AMP03_PS_PF_V5,PP_AV_AMP03_TL_PF_V5,PP_AV_GRV01_IF_PF_V5,PP_AV_GRV01_PS_PF_V5,PP_AV_GRV01_TL_PF_V5,PP_AV_GRV02_IF_PF_V5,PP_AV_GRV02_PS_PF_V5,PP_AV_GRV02_TL_PF_V5,PP_AV_GRV03_IF_PF_V5,PP_AV_GRV03_PS_PF_V5,PP_AV_GRV03_TL_PF_V5,PP_AV_GRV04_IF_PF_V5,PP_AV_GRV04_PS_PF_V5,PP_AV_GRV04_TL_PF_V5,PP_AV_GRV05_IF_PF_V5,PP_AV_GRV05_PS_PF_V5,PP_AV_GRV05_TL_PF_V5,PP_AV_GRV06_IF_PF_V5,PP_AV_GRV06_PS_PF_V5,PP_AV_GRV06_TL_PF_V5,PP_AV_MRC01_IF_PF_V5,PP_AV_MRC01_PS_PF_V5,PP_AV_MRC01_TL_PF_V5,PP_AV_MRC02_IF_PF_V5,PP_AV_MRC02_PS_PF_V5,PP_AV_MRC02_TL_PF_V5,PP_AV_MRC03_IF_PF_V5,PP_AV_MRC03_PS_PF_V5,PP_AV_MRC03_TL_PF_V5,PP_AV_MRC04_IF_PF_V5,PP_AV_MRC04_PS_PF_V5,PP_AV_MRC04_TL_PF_V5,PP_AV_TDC01_IF_PF_V5,PP_AV_TDC01_PS_PF_V5,PP_AV_TDC01_TL_PF_V5,PP_AV_TDC02_IF_PF_V5,PP_AV_TDC02_PS_PF_V5,PP_AV_TDC02_TL_PF_V5,PP_HA_AMP01_IF_PF_V5,PP_HA_AMP01_PS_PF_V5,PP_HA_AMP01_TL_PF_V5,PP_HA_AMP02_IF_PF_V5,PP_HA_AMP02_PS_PF_V5,PP_HA_AMP02_TL_PF_V5,PP_HA_FRQ01_IF_PF_V5,PP_HA_FRQ01_PS_PF_V5,PP_HA_FRQ01_TL_PF_V5,PP_HA_REC01_IF_PF_V5,PP_HA_REC01_PS_PF_V5,PP_HA_REC01_TL_PF_V5,PP_HA_VLR01_IF_PF_V5,PP_HA_VLR01_PS_PF_V5,PP_HA_VLR01_TL_PF_V5,PP_HA_VLR02_IF_PF_V5,PP_HA_VLR02_PS_PF_V5,PP_HA_VLR02_TL_PF_V5,PP_PA_VLR01_IF_PF_V5,PP_PA_VLR01_PS_PF_V5,PP_PA_VLR01_TL_PF_V5,PP_PD_CPG01_IF_PF_V5,PP_PD_CPG01_PS_PF_V5,PP_PD_CPG01_TL_PF_V5,PP_PD_CPG02_IF_PF_V5,PP_PD_CPG02_PS_PF_V5,PP_PD_CPG02_TL_PF_V5,PP_PD_VPG01_IF_PF_V5,PP_PD_VPG01_PS_PF_V5,PP_PD_VPG01_TL_PF_V5,RM_BC_FRQ01_IF_PF_V5,RM_BC_FRQ01_PS_PF_V5,RM_BC_FRQ01_TL_PF_V5,RM_BC_REC01_IF_PF_V5,RM_BC_REC01_PS_PF_V5,RM_BC_REC01_TL_PF_V5,RM_BC_REC02_IF_PF_V5,RM_PR_AMP01_IF_PF_V5,RM_PR_AMP01_PS_PF_V5,RM_PR_AMP01_TL_PF_V5,RM_PR_CAR01_TL_V5,RM_PR_CAR02_TL_V5,RM_PR_CAR03_TL_V5,RM_PR_CAR04_TL_V5,RM_PR_CAR05_TL_V5,RM_PR_CAR06_TL_V5,RM_PR_CAR07_TL_V5,RM_PR_CAR08_TL_V5,RM_PR_CAR09_TL_V5,RM_PR_CAR10_TL_V5,RM_PR_CAR11_TL_V5,RM_PR_CAR12_TL_V5,RM_PR_CNC01_IF_PF_V5,RM_PR_CNC01_PS_PF_V5,RM_PR_CNC01_TL_PF_V5,RM_PR_CNC02_IF_PF_V5,RM_PR_CNC02_PS_PF_V5,RM_PR_CNC02_TL_PF_V5,RM_PR_DCP01_PS_PF_V5,RM_PR_ETB01_IF_PF_V5,RM_PR_ETB01_PS_PF_V5,RM_PR_ETB01_TL_PF_V5,RM_PR_ETB02_IF_PF_V5,RM_PR_ETB02_PS_PF_V5,RM_PR_ETB02_TL_PF_V5,RM_PR_MOD01_IF_PF_V5,RM_PR_MOD02_IF_PF_V5,RM_PR_MOD03_IF_PF_V5,RM_PR_MOD04_IF_PF_V5,RM_PR_MOD07_TL_PF_V5
0,10092403581,1795916,2025-01-14,2025-01,NaN,NaN,NaN,aprovado,dentro,0,DESKTOP_PRONTO_PF_DIGITAL,0,758,0,NaN,0,NaN,1,1,0,0,0,"[740, 760)",0.0,0.0,0,0.0,0.00,304.00,304.0,24.49,24.49,304.00,0.57,0.57,304.00,59.36,59.36,304.00,304.0,304.00,304.00,10.52,10.52,304.00,304.0,0.0,304.0,304.0,304.0,304.0,304.0,304.00,304.00,48.36,48.36,304.00,0.0,26.83,15.86,15.86,1.95,76.16,76.16,304.0,0.0,0.0,304.0,76.16,76.16,304.0,0.0,0.0,304.0,0.0,0.0,304.0,100.0,100.0,304.0,0.0,0.0,304.0,0.0,0.0,304.0,0.0,0.0,304.0,0

**Retificar a coluna cpf**

In [5]:
df['cpf'] = df['cpf'].str.replace(r'\.0$', '', regex=True).str.zfill(11)

## ETL dos indicadores

* O range de valores que os indicadores da QUOD podem ter está definido entre 0 a 100.
* Fora deste intervalo, é código.

In [6]:
def processar_df(df_original):
    # Crie uma cópia do DataFrame original para não modificá-lo
    df_novo = df_original.copy()
    
    # Encontre as colunas que correspondem ao padrão "DUAS_LETRAS_MAIÚSCULAS_ + _"
    # O r'' significa que é uma string raw, o que facilita usar a barra '\'
    # O ^ significa que o padrão deve começar no início da string
    # [A-Z]{2} procura exatamente duas letras maiúsculas
    # _ procura o underscore
    # O .*(?) procura o resto da string de forma não gananciosa
    colunas_para_processar = [col for col in df_novo.columns if col[0:2].isupper() and col[2] == '_']
    
    # Itere sobre as colunas encontradas
    for coluna in colunas_para_processar:
        # Pega a série de dados da coluna
        serie = df_novo[coluna]
        
        # Converte a coluna para tipo numérico, forçando erros para NaN
        serie_numerica = pd.to_numeric(serie, errors='coerce')
        
        # Cria uma máscara booleana para encontrar os valores que NÃO estão no intervalo (0-100)
        # O | significa OR (ou)
        mascara_fora_intervalo = (serie_numerica < 0) | (serie_numerica > 100)
        
        # Usa .loc pra substituir os valores que correspondem à máscara por NaN
        # O .loc garante que a operação é segura e evita o aviso "SettingWithCopyWarning"
        df_novo.loc[mascara_fora_intervalo, coluna] = np.nan
        
    return df_novo

# Exemplo de uso (crie seu próprio DataFrame aqui)
# Suponha que o seu DataFrame original se chame 'df_meus_dados'
# df_novo = processar_df(df_meus_dados)
# print(df_novo)

In [7]:
df_novo = processar_df(df)
#Visualização
df_novo.head(3)

,cpf,num_adm,ref_date,safra,data_instalacao,data_cancelamento,dias_empresa,status_aplicacao,dentro_politica,entrada_tv,canal_aquisicao,flag_canal_fisico,quod_score,fpd,flag_churn,churn_curto_prazo,motivo,segmento_quod,segmento_quod_1,segmento_quod_2,segmento_quod_3,segmento_quod_4,escore_faixa,quod_total_restritivos,quod_valor_restritivos,quod_flag_restritivo,GC_AV_GRV01_IF_PF_V5,GC_AV_GRV01_PS_PF_V5,GC_AV_GRV01_TL_PF_V5,GC_EM_ITS01_IF_PF_V5,GC_PD_REC01_IF_PF_V5,GC_PD_REC01_PS_PF_V5,GC_PD_REC01_TL_PF_V5,PF_CP_CNT01_IF_PF_V5,PF_CP_CNT01_PS_PF_V5,PF_CP_CNT01_TL_PF_V5,PF_CP_CNT02_IF_PF_V5,PF_CP_CNT02_PS_PF_V5,PF_CP_CNT02_TL_PF_V5,PF_CP_VOL01_IF_PF_V5,PF_CP_VOL01_PS_PF_V5,PF_CP_VOL01_TL_PF_V5,PF_CP_VOL02_IF_PF_V5,PF_CP_VOL02_PS_PF_V5,PF_CP_VOL02_TL_PF_V5,PF_CP_VOL03_IF_PF_V5,PF_CP_VOL04_IF_PF_V5,PF_EM_HBT01_IF_PF_V5,PF_EM_HBT02_IF_PF_V5,PF_EM_ITS01_IF_PF_V5,PF_EM_ITS02_IF_PF_V5,PF_PC_CPR01_IF_PF_V5,PF_PC_CPR01_PS_PF_V5,PF_PC_CPR01_TL_PF_V5,PF_PC_CPR02_IF_PF_V5,PF_PC_CPR02_PS_PF_V5,PF_PC_CPR02_TL_PF_V5,PF_PC_PRD01_IF_PF_V5,PF_PC_PRD02_IF_PF_V5,PF_PC_TPR01_IF_PF_V5,PF_PC_TPR02_IF_PF_V5,PF_PC_VLR01_IF_PF_V5,PP_AV_AMP01_IF_PF_V5,PP_AV_AMP01_PS_PF_V5,PP_AV_AMP01_TL_PF_V5,PP_AV_AMP02_IF_PF_V5,PP_AV_AMP02_PS_PF_V5,PP_AV_AMP02_TL_PF_V5,PP_AV_AMP03_IF_PF_V5,PP_AV_AMP03_PS_PF_V5,PP_AV_AMP03_TL_PF_V5,PP_AV_GRV01_IF_PF_V5,PP_AV_GRV01_PS_PF_V5,PP_AV_GRV01_TL_PF_V5,PP_AV_GRV02_IF_PF_V5,PP_AV_GRV02_PS_PF_V5,PP_AV_GRV02_TL_PF_V5,PP_AV_GRV03_IF_PF_V5,PP_AV_GRV03_PS_PF_V5,PP_AV_GRV03_TL_PF_V5,PP_AV_GRV04_IF_PF_V5,PP_AV_GRV04_PS_PF_V5,PP_AV_GRV04_TL_PF_V5,PP_AV_GRV05_IF_PF_V5,PP_AV_GRV05_PS_PF_V5,PP_AV_GRV05_TL_PF_V5,PP_AV_GRV06_IF_PF_V5,PP_AV_GRV06_PS_PF_V5,PP_AV_GRV06_TL_PF_V5,PP_AV_MRC01_IF_PF_V5,PP_AV_MRC01_PS_PF_V5,PP_AV_MRC01_TL_PF_V5,PP_AV_MRC02_IF_PF_V5,PP_AV_MRC02_PS_PF_V5,PP_AV_MRC02_TL_PF_V5,PP_AV_MRC03_IF_PF_V5,PP_AV_MRC03_PS_PF_V5,PP_AV_MRC03_TL_PF_V5,PP_AV_MRC04_IF_PF_V5,PP_AV_MRC04_PS_PF_V5,PP_AV_MRC04_TL_PF_V5,PP_AV_TDC01_IF_PF_V5,PP_AV_TDC01_PS_PF_V5,PP_AV_TDC01_TL_PF_V5,PP_AV_TDC02_IF_PF_V5,PP_AV_TDC02_PS_PF_V5,PP_AV_TDC02_TL_PF_V5,PP_HA_AMP01_IF_PF_V5,PP_HA_AMP01_PS_PF_V5,PP_HA_AMP01_TL_PF_V5,PP_HA_AMP02_IF_PF_V5,PP_HA_AMP02_PS_PF_V5,PP_HA_AMP02_TL_PF_V5,PP_HA_FRQ01_IF_PF_V5,PP_HA_FRQ01_PS_PF_V5,PP_HA_FRQ01_TL_PF_V5,PP_HA_REC01_IF_PF_V5,PP_HA_REC01_PS_PF_V5,PP_HA_REC01_TL_PF_V5,PP_HA_VLR01_IF_PF_V5,PP_HA_VLR01_PS_PF_V5,PP_HA_VLR01_TL_PF_V5,PP_HA_VLR02_IF_PF_V5,PP_HA_VLR02_PS_PF_V5,PP_HA_VLR02_TL_PF_V5,PP_PA_VLR01_IF_PF_V5,PP_PA_VLR01_PS_PF_V5,PP_PA_VLR01_TL_PF_V5,PP_PD_CPG01_IF_PF_V5,PP_PD_CPG01_PS_PF_V5,PP_PD_CPG01_TL_PF_V5,PP_PD_CPG02_IF_PF_V5,PP_PD_CPG02_PS_PF_V5,PP_PD_CPG02_TL_PF_V5,PP_PD_VPG01_IF_PF_V5,PP_PD_VPG01_PS_PF_V5,PP_PD_VPG01_TL_PF_V5,RM_BC_FRQ01_IF_PF_V5,RM_BC_FRQ01_PS_PF_V5,RM_BC_FRQ01_TL_PF_V5,RM_BC_REC01_IF_PF_V5,RM_BC_REC01_PS_PF_V5,RM_BC_REC01_TL_PF_V5,RM_BC_REC02_IF_PF_V5,RM_PR_AMP01_IF_PF_V5,RM_PR_AMP01_PS_PF_V5,RM_PR_AMP01_TL_PF_V5,RM_PR_CAR01_TL_V5,RM_PR_CAR02_TL_V5,RM_PR_CAR03_TL_V5,RM_PR_CAR04_TL_V5,RM_PR_CAR05_TL_V5,RM_PR_CAR06_TL_V5,RM_PR_CAR07_TL_V5,RM_PR_CAR08_TL_V5,RM_PR_CAR09_TL_V5,RM_PR_CAR10_TL_V5,RM_PR_CAR11_TL_V5,RM_PR_CAR12_TL_V5,RM_PR_CNC01_IF_PF_V5,RM_PR_CNC01_PS_PF_V5,RM_PR_CNC01_TL_PF_V5,RM_PR_CNC02_IF_PF_V5,RM_PR_CNC02_PS_PF_V5,RM_PR_CNC02_TL_PF_V5,RM_PR_DCP01_PS_PF_V5,RM_PR_ETB01_IF_PF_V5,RM_PR_ETB01_PS_PF_V5,RM_PR_ETB01_TL_PF_V5,RM_PR_ETB02_IF_PF_V5,RM_PR_ETB02_PS_PF_V5,RM_PR_ETB02_TL_PF_V5,RM_PR_MOD01_IF_PF_V5,RM_PR_MOD02_IF_PF_V5,RM_PR_MOD03_IF_PF_V5,RM_PR_MOD04_IF_PF_V5,RM_PR_MOD07_TL_PF_V5
0,10092403581,1795916,2025-01-14,2025-01,NaN,NaN,NaN,aprovado,dentro,0,DESKTOP_PRONTO_PF_DIGITAL,0,758,0,NaN,0,NaN,1,1,0,0,0,"[740, 760)",0.0,0.0,0,0.0,0.00,NaN,NaN,24.49,24.49,NaN,0.57,0.57,NaN,59.36,59.36,NaN,NaN,NaN,NaN,10.52,10.52,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.36,48.36,NaN,0.0,26.83,15.86,15.86,1.95,76.16,76.16,NaN,0.0,0.0,NaN,76.16,76.16,NaN,0.0,0.0,NaN,0.0,0.0,NaN,100.0,100.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,98.2,98.20,NaN,10

In [8]:
df_novo_2 = df_novo.drop([ 'data_instalacao', 'data_cancelamento', 'dias_empresa',
                         'flag_churn', 'churn_curto_prazo', 'churn_curto_prazo', 'motivo'], axis=1)

In [9]:
#Exportar para CSV
df_novo_2.to_csv('base_quod_indicadores_transformada.csv', header=True, index=False, sep=';')

# 2 - Filtrar os indicadores da QUOD e o fpd

In [10]:
#Seleção das colunas numéricas
numeric_cols = df_novo.select_dtypes(include=['int64', 'float64']).columns #Filtrar as colunas cujo tipo é inteiro ou float
#Data frame das variáveis numéricas
df_2 = df_novo[numeric_cols]
#Lista de colunas a serem removidas 
cols_to_drop = ['dias_empresa', 'quod_score', 'flag_churn', 'churn_curto_prazo',
               'segmento_quod', 'segmento_quod_1', 'segmento_quod_2', 'segmento_quod_3', 'segmento_quod_4',
               'quod_total_restritivos', 'quod_valor_restritivos', 'quod_flag_restritivo', 'entrada_tv', 'flag_canal_fisico'] 
#Remover as colunas
df_selecao_inicial = df_2.drop(columns=cols_to_drop)
#Visualização do data frame das colunas numéricas
df_selecao_inicial.head(3)

,fpd,GC_AV_GRV01_IF_PF_V5,GC_AV_GRV01_PS_PF_V5,GC_AV_GRV01_TL_PF_V5,GC_EM_ITS01_IF_PF_V5,GC_PD_REC01_IF_PF_V5,GC_PD_REC01_PS_PF_V5,GC_PD_REC01_TL_PF_V5,PF_CP_CNT01_IF_PF_V5,PF_CP_CNT01_PS_PF_V5,PF_CP_CNT01_TL_PF_V5,PF_CP_CNT02_IF_PF_V5,PF_CP_CNT02_PS_PF_V5,PF_CP_CNT02_TL_PF_V5,PF_CP_VOL01_IF_PF_V5,PF_CP_VOL01_PS_PF_V5,PF_CP_VOL01_TL_PF_V5,PF_CP_VOL02_IF_PF_V5,PF_CP_VOL02_PS_PF_V5,PF_CP_VOL02_TL_PF_V5,PF_CP_VOL03_IF_PF_V5,PF_CP_VOL04_IF_PF_V5,PF_EM_HBT01_IF_PF_V5,PF_EM_HBT02_IF_PF_V5,PF_EM_ITS01_IF_PF_V5,PF_EM_ITS02_IF_PF_V5,PF_PC_CPR01_IF_PF_V5,PF_PC_CPR01_PS_PF_V5,PF_PC_CPR01_TL_PF_V5,PF_PC_CPR02_IF_PF_V5,PF_PC_CPR02_PS_PF_V5,PF_PC_CPR02_TL_PF_V5,PF_PC_PRD01_IF_PF_V5,PF_PC_PRD02_IF_PF_V5,PF_PC_TPR01_IF_PF_V5,PF_PC_TPR02_IF_PF_V5,PF_PC_VLR01_IF_PF_V5,PP_AV_AMP01_IF_PF_V5,PP_AV_AMP01_PS_PF_V5,PP_AV_AMP01_TL_PF_V5,PP_AV_AMP02_IF_PF_V5,PP_AV_AMP02_PS_PF_V5,PP_AV_AMP02_TL_PF_V5,PP_AV_AMP03_IF_PF_V5,PP_AV_AMP03_PS_PF_V5,PP_AV_AMP03_TL_PF_V5,PP_AV_GRV01_IF_PF_V5,PP_AV_GRV01_PS_PF_V5,PP_AV_GRV01_TL_PF_V5,PP_AV_GRV02_IF_PF_V5,PP_AV_GRV02_PS_PF_V5,PP_AV_GRV02_TL_PF_V5,PP_AV_GRV03_IF_PF_V5,PP_AV_GRV03_PS_PF_V5,PP_AV_GRV03_TL_PF_V5,PP_AV_GRV04_IF_PF_V5,PP_AV_GRV04_PS_PF_V5,PP_AV_GRV04_TL_PF_V5,PP_AV_GRV05_IF_PF_V5,PP_AV_GRV05_PS_PF_V5,PP_AV_GRV05_TL_PF_V5,PP_AV_GRV06_IF_PF_V5,PP_AV_GRV06_PS_PF_V5,PP_AV_GRV06_TL_PF_V5,PP_AV_MRC01_IF_PF_V5,PP_AV_MRC01_PS_PF_V5,PP_AV_MRC01_TL_PF_V5,PP_AV_MRC02_IF_PF_V5,PP_AV_MRC02_PS_PF_V5,PP_AV_MRC02_TL_PF_V5,PP_AV_MRC03_IF_PF_V5,PP_AV_MRC03_PS_PF_V5,PP_AV_MRC03_TL_PF_V5,PP_AV_MRC04_IF_PF_V5,PP_AV_MRC04_PS_PF_V5,PP_AV_MRC04_TL_PF_V5,PP_AV_TDC01_IF_PF_V5,PP_AV_TDC01_PS_PF_V5,PP_AV_TDC01_TL_PF_V5,PP_AV_TDC02_IF_PF_V5,PP_AV_TDC02_PS_PF_V5,PP_AV_TDC02_TL_PF_V5,PP_HA_AMP01_IF_PF_V5,PP_HA_AMP01_PS_PF_V5,PP_HA_AMP01_TL_PF_V5,PP_HA_AMP02_IF_PF_V5,PP_HA_AMP02_PS_PF_V5,PP_HA_AMP02_TL_PF_V5,PP_HA_FRQ01_IF_PF_V5,PP_HA_FRQ01_PS_PF_V5,PP_HA_FRQ01_TL_PF_V5,PP_HA_REC01_IF_PF_V5,PP_HA_REC01_PS_PF_V5,PP_HA_REC01_TL_PF_V5,PP_HA_VLR01_IF_PF_V5,PP_HA_VLR01_PS_PF_V5,PP_HA_VLR01_TL_PF_V5,PP_HA_VLR02_IF_PF_V5,PP_HA_VLR02_PS_PF_V5,PP_HA_VLR02_TL_PF_V5,PP_PA_VLR01_IF_PF_V5,PP_PA_VLR01_PS_PF_V5,PP_PA_VLR01_TL_PF_V5,PP_PD_CPG01_IF_PF_V5,PP_PD_CPG01_PS_PF_V5,PP_PD_CPG01_TL_PF_V5,PP_PD_CPG02_IF_PF_V5,PP_PD_CPG02_PS_PF_V5,PP_PD_CPG02_TL_PF_V5,PP_PD_VPG01_IF_PF_V5,PP_PD_VPG01_PS_PF_V5,PP_PD_VPG01_TL_PF_V5,RM_BC_FRQ01_IF_PF_V5,RM_BC_FRQ01_PS_PF_V5,RM_BC_FRQ01_TL_PF_V5,RM_BC_REC01_IF_PF_V5,RM_BC_REC01_PS_PF_V5,RM_BC_REC01_TL_PF_V5,RM_BC_REC02_IF_PF_V5,RM_PR_AMP01_IF_PF_V5,RM_PR_AMP01_PS_PF_V5,RM_PR_AMP01_TL_PF_V5,RM_PR_CAR01_TL_V5,RM_PR_CAR02_TL_V5,RM_PR_CAR03_TL_V5,RM_PR_CAR04_TL_V5,RM_PR_CAR05_TL_V5,RM_PR_CAR06_TL_V5,RM_PR_CAR07_TL_V5,RM_PR_CAR08_TL_V5,RM_PR_CAR09_TL_V5,RM_PR_CAR10_TL_V5,RM_PR_CAR11_TL_V5,RM_PR_CAR12_TL_V5,RM_PR_CNC01_IF_PF_V5,RM_PR_CNC01_PS_PF_V5,RM_PR_CNC01_TL_PF_V5,RM_PR_CNC02_IF_PF_V5,RM_PR_CNC02_PS_PF_V5,RM_PR_CNC02_TL_PF_V5,RM_PR_DCP01_PS_PF_V5,RM_PR_ETB01_IF_PF_V5,RM_PR_ETB01_PS_PF_V5,RM_PR_ETB01_TL_PF_V5,RM_PR_ETB02_IF_PF_V5,RM_PR_ETB02_PS_PF_V5,RM_PR_ETB02_TL_PF_V5,RM_PR_MOD01_IF_PF_V5,RM_PR_MOD02_IF_PF_V5,RM_PR_MOD03_IF_PF_V5,RM_PR_MOD04_IF_PF_V5,RM_PR_MOD07_TL_PF_V5
0,0,0.0,0.00,NaN,NaN,24.49,24.49,NaN,0.57,0.57,NaN,59.36,59.36,NaN,NaN,NaN,NaN,10.52,10.52,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.36,48.36,NaN,0.0,26.83,15.86,15.86,1.95,76.16,76.16,NaN,0.0,0.0,NaN,76.16,76.16,NaN,0.0,0.0,NaN,0.0,0.0,NaN,100.0,100.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,98.2,98.20,NaN,100.0,100.00,NaN,70.39,70.39,NaN,42.19,42.19,NaN,100.0,100.00,NaN,100.0,100.00,NaN,0.0,0.00,NaN,NaN,NaN,NaN,21.26,21.26,NaN,71.43,71.43,NaN,35.84,35.84,NaN,58.08,58.08,NaN,32.15,32.15,NaN,0.0,0.0,NaN,29.13,84.83,84.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.05,1.05,NaN,27.03,27.03,NaN,75.0,99.42,99.42,NaN,49.53,49.53,NaN,41.67,100.0,NaN,100.00,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96.60,96.60,NaN,99.53,99.53,NaN,0.0,0.00,NaN,0.00,0.00,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.00,NaN,0.00,0.00,N

In [11]:
#Data frame das features
features_quantitativas = df_selecao_inicial.drop(columns=['fpd']) # Todas as colunas exceto 'ALVO'
#Data frame do ALVO
y = df_selecao_inicial['fpd'] 

# 3 - Correlação entre o fpd e os indicadores

* A correlação de spearman analisa a relação monotônica força e direção) entre duas variáveis.

In [12]:
#Matriz de correlação
correlacoes_spearman = df_selecao_inicial.corr(method='spearman')

In [13]:
#Filtrar as correlações apenas entre o fpd e os indicadores
correlacao_com_alvo = correlacoes_spearman['fpd']
#Remover a coluna fpd
correlacao_final = correlacao_com_alvo.drop('fpd')
#Transformar num data frame
df_correlacao = correlacao_final.to_frame()

# 1. Transforma o índice em coluna
df_final = df_correlacao.reset_index()

# 2. Renomeia as colunas para o formato desejado
df_final.columns = ['VARIAVEL', 'CORRELACAO_ALVO']
###################
#Visualização
###################
df_final.head(3)

,VARIAVEL,CORRELACAO_ALVO
0,GC_AV_GRV01_IF_PF_V5,0.083314
1,GC_AV_GRV01_PS_PF_V5,0.071916
2,GC_AV_GRV01_TL_PF_V5,0.063265


# 4 - Correlação entre o escore e os indicadores

In [14]:
#Data frame das variáveis numéricas
df_2 = df_novo[numeric_cols]
#Lista de colunas a serem removidas 
cols_to_drop_v2 = ['dias_empresa', 'flag_churn', 'churn_curto_prazo', 'fpd',
               'segmento_quod', 'segmento_quod_1', 'segmento_quod_2', 'segmento_quod_3', 'segmento_quod_4',
               'quod_total_restritivos', 'quod_valor_restritivos', 'quod_flag_restritivo', 'entrada_tv', 'flag_canal_fisico'] 
#Remover as colunas
df_selecao_inicial_v2 = df_2.drop(columns=cols_to_drop_v2)
#Visualização do data frame das colunas numéricas
df_selecao_inicial_v2.head(3)

,quod_score,GC_AV_GRV01_IF_PF_V5,GC_AV_GRV01_PS_PF_V5,GC_AV_GRV01_TL_PF_V5,GC_EM_ITS01_IF_PF_V5,GC_PD_REC01_IF_PF_V5,GC_PD_REC01_PS_PF_V5,GC_PD_REC01_TL_PF_V5,PF_CP_CNT01_IF_PF_V5,PF_CP_CNT01_PS_PF_V5,PF_CP_CNT01_TL_PF_V5,PF_CP_CNT02_IF_PF_V5,PF_CP_CNT02_PS_PF_V5,PF_CP_CNT02_TL_PF_V5,PF_CP_VOL01_IF_PF_V5,PF_CP_VOL01_PS_PF_V5,PF_CP_VOL01_TL_PF_V5,PF_CP_VOL02_IF_PF_V5,PF_CP_VOL02_PS_PF_V5,PF_CP_VOL02_TL_PF_V5,PF_CP_VOL03_IF_PF_V5,PF_CP_VOL04_IF_PF_V5,PF_EM_HBT01_IF_PF_V5,PF_EM_HBT02_IF_PF_V5,PF_EM_ITS01_IF_PF_V5,PF_EM_ITS02_IF_PF_V5,PF_PC_CPR01_IF_PF_V5,PF_PC_CPR01_PS_PF_V5,PF_PC_CPR01_TL_PF_V5,PF_PC_CPR02_IF_PF_V5,PF_PC_CPR02_PS_PF_V5,PF_PC_CPR02_TL_PF_V5,PF_PC_PRD01_IF_PF_V5,PF_PC_PRD02_IF_PF_V5,PF_PC_TPR01_IF_PF_V5,PF_PC_TPR02_IF_PF_V5,PF_PC_VLR01_IF_PF_V5,PP_AV_AMP01_IF_PF_V5,PP_AV_AMP01_PS_PF_V5,PP_AV_AMP01_TL_PF_V5,PP_AV_AMP02_IF_PF_V5,PP_AV_AMP02_PS_PF_V5,PP_AV_AMP02_TL_PF_V5,PP_AV_AMP03_IF_PF_V5,PP_AV_AMP03_PS_PF_V5,PP_AV_AMP03_TL_PF_V5,PP_AV_GRV01_IF_PF_V5,PP_AV_GRV01_PS_PF_V5,PP_AV_GRV01_TL_PF_V5,PP_AV_GRV02_IF_PF_V5,PP_AV_GRV02_PS_PF_V5,PP_AV_GRV02_TL_PF_V5,PP_AV_GRV03_IF_PF_V5,PP_AV_GRV03_PS_PF_V5,PP_AV_GRV03_TL_PF_V5,PP_AV_GRV04_IF_PF_V5,PP_AV_GRV04_PS_PF_V5,PP_AV_GRV04_TL_PF_V5,PP_AV_GRV05_IF_PF_V5,PP_AV_GRV05_PS_PF_V5,PP_AV_GRV05_TL_PF_V5,PP_AV_GRV06_IF_PF_V5,PP_AV_GRV06_PS_PF_V5,PP_AV_GRV06_TL_PF_V5,PP_AV_MRC01_IF_PF_V5,PP_AV_MRC01_PS_PF_V5,PP_AV_MRC01_TL_PF_V5,PP_AV_MRC02_IF_PF_V5,PP_AV_MRC02_PS_PF_V5,PP_AV_MRC02_TL_PF_V5,PP_AV_MRC03_IF_PF_V5,PP_AV_MRC03_PS_PF_V5,PP_AV_MRC03_TL_PF_V5,PP_AV_MRC04_IF_PF_V5,PP_AV_MRC04_PS_PF_V5,PP_AV_MRC04_TL_PF_V5,PP_AV_TDC01_IF_PF_V5,PP_AV_TDC01_PS_PF_V5,PP_AV_TDC01_TL_PF_V5,PP_AV_TDC02_IF_PF_V5,PP_AV_TDC02_PS_PF_V5,PP_AV_TDC02_TL_PF_V5,PP_HA_AMP01_IF_PF_V5,PP_HA_AMP01_PS_PF_V5,PP_HA_AMP01_TL_PF_V5,PP_HA_AMP02_IF_PF_V5,PP_HA_AMP02_PS_PF_V5,PP_HA_AMP02_TL_PF_V5,PP_HA_FRQ01_IF_PF_V5,PP_HA_FRQ01_PS_PF_V5,PP_HA_FRQ01_TL_PF_V5,PP_HA_REC01_IF_PF_V5,PP_HA_REC01_PS_PF_V5,PP_HA_REC01_TL_PF_V5,PP_HA_VLR01_IF_PF_V5,PP_HA_VLR01_PS_PF_V5,PP_HA_VLR01_TL_PF_V5,PP_HA_VLR02_IF_PF_V5,PP_HA_VLR02_PS_PF_V5,PP_HA_VLR02_TL_PF_V5,PP_PA_VLR01_IF_PF_V5,PP_PA_VLR01_PS_PF_V5,PP_PA_VLR01_TL_PF_V5,PP_PD_CPG01_IF_PF_V5,PP_PD_CPG01_PS_PF_V5,PP_PD_CPG01_TL_PF_V5,PP_PD_CPG02_IF_PF_V5,PP_PD_CPG02_PS_PF_V5,PP_PD_CPG02_TL_PF_V5,PP_PD_VPG01_IF_PF_V5,PP_PD_VPG01_PS_PF_V5,PP_PD_VPG01_TL_PF_V5,RM_BC_FRQ01_IF_PF_V5,RM_BC_FRQ01_PS_PF_V5,RM_BC_FRQ01_TL_PF_V5,RM_BC_REC01_IF_PF_V5,RM_BC_REC01_PS_PF_V5,RM_BC_REC01_TL_PF_V5,RM_BC_REC02_IF_PF_V5,RM_PR_AMP01_IF_PF_V5,RM_PR_AMP01_PS_PF_V5,RM_PR_AMP01_TL_PF_V5,RM_PR_CAR01_TL_V5,RM_PR_CAR02_TL_V5,RM_PR_CAR03_TL_V5,RM_PR_CAR04_TL_V5,RM_PR_CAR05_TL_V5,RM_PR_CAR06_TL_V5,RM_PR_CAR07_TL_V5,RM_PR_CAR08_TL_V5,RM_PR_CAR09_TL_V5,RM_PR_CAR10_TL_V5,RM_PR_CAR11_TL_V5,RM_PR_CAR12_TL_V5,RM_PR_CNC01_IF_PF_V5,RM_PR_CNC01_PS_PF_V5,RM_PR_CNC01_TL_PF_V5,RM_PR_CNC02_IF_PF_V5,RM_PR_CNC02_PS_PF_V5,RM_PR_CNC02_TL_PF_V5,RM_PR_DCP01_PS_PF_V5,RM_PR_ETB01_IF_PF_V5,RM_PR_ETB01_PS_PF_V5,RM_PR_ETB01_TL_PF_V5,RM_PR_ETB02_IF_PF_V5,RM_PR_ETB02_PS_PF_V5,RM_PR_ETB02_TL_PF_V5,RM_PR_MOD01_IF_PF_V5,RM_PR_MOD02_IF_PF_V5,RM_PR_MOD03_IF_PF_V5,RM_PR_MOD04_IF_PF_V5,RM_PR_MOD07_TL_PF_V5
0,758,0.0,0.00,NaN,NaN,24.49,24.49,NaN,0.57,0.57,NaN,59.36,59.36,NaN,NaN,NaN,NaN,10.52,10.52,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.36,48.36,NaN,0.0,26.83,15.86,15.86,1.95,76.16,76.16,NaN,0.0,0.0,NaN,76.16,76.16,NaN,0.0,0.0,NaN,0.0,0.0,NaN,100.0,100.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,98.2,98.20,NaN,100.0,100.00,NaN,70.39,70.39,NaN,42.19,42.19,NaN,100.0,100.00,NaN,100.0,100.00,NaN,0.0,0.00,NaN,NaN,NaN,NaN,21.26,21.26,NaN,71.43,71.43,NaN,35.84,35.84,NaN,58.08,58.08,NaN,32.15,32.15,NaN,0.0,0.0,NaN,29.13,84.83,84.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.05,1.05,NaN,27.03,27.03,NaN,75.0,99.42,99.42,NaN,49.53,49.53,NaN,41.67,100.0,NaN,100.00,NaN
1,684,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96.60,96.60,NaN,99.53,99.53,NaN,0.0,0.00,NaN,0.00,0.00,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.00,NaN,

In [15]:
#Matriz de correlação
correlacoes_spearman_v2 = df_selecao_inicial_v2.corr(method='spearman')

In [16]:
#Filtrar as correlações apenas entre o fpd e os indicadores
correlacao_com_escore = correlacoes_spearman_v2['quod_score']
#Remover a coluna fpd
correlacao_final_escore = correlacao_com_escore.drop('quod_score')
#Transformar num data frame
df_correlacao_escore = correlacao_final_escore.to_frame()

# 1. Transforma o índice em coluna
df_final_escore = df_correlacao_escore.reset_index()

# 2. Renomeia as colunas para o formato desejado
df_final_escore.columns = ['VARIAVEL', 'CORRELACAO_ESCORE_INDICADOR']
###################
#Visualização
###################
df_final_escore.head(3)

,VARIAVEL,CORRELACAO_ESCORE_INDICADOR
0,GC_AV_GRV01_IF_PF_V5,-0.460914
1,GC_AV_GRV01_PS_PF_V5,-0.393326
2,GC_AV_GRV01_TL_PF_V5,-0.412440


# 5 - Estatística Descritiva

In [17]:
# 1. Calcular o percentual de missing
missing_percent = features_quantitativas.isna().mean() * 100

# 2. Criar o dicionário de estatísticas, incluindo o percentual de missing
estatisticas = {
    'MEDIA': features_quantitativas.mean(),
    'MEDIANA': features_quantitativas.median(),
    'MAXIMO': features_quantitativas.max(),
    'MINIMO': features_quantitativas.min(),
    'P_25': features_quantitativas.quantile(0.25),
    'P_75': features_quantitativas.quantile(0.75),
    'DESVIO_PADRAO': features_quantitativas.std(),
    'COEFICIENTE_VARIACAO': (features_quantitativas.std() / features_quantitativas.mean()) * 100,
    'PERCENTUAL_MISSING': missing_percent
}

# 3. Criar DataFrame e formatar nomes das variáveis em MAIÚSCULAS
df_estatisticas = pd.DataFrame(estatisticas)
df_estatisticas.index = df_estatisticas.index.str.upper()

# 4. Resetar índice para transformar 'VARIAVEL' em coluna
df_estatisticas = df_estatisticas.reset_index().rename(columns={'index': 'VARIAVEL'})

# 5. Ordenar colunas, incluindo 'PERCENTUAL_MISSING'
colunas_ordenadas = ['VARIAVEL', 'PERCENTUAL_MISSING', 'MEDIA', 'MEDIANA', 'DESVIO_PADRAO', 'COEFICIENTE_VARIACAO',
                     'MINIMO', 'P_25', 'P_75', 'MAXIMO']
df_estatisticas = df_estatisticas[colunas_ordenadas]

# 6. Mostrar resultado
df_estatisticas.head(3)

,VARIAVEL,PERCENTUAL_MISSING,MEDIA,MEDIANA,DESVIO_PADRAO,COEFICIENTE_VARIACAO,MINIMO,P_25,P_75,MAXIMO
0,GC_AV_GRV01_IF_PF_V5,26.706065,27.360325,8.57,33.205314,121.363010,0.0,0.0,53.63,100.0
1,GC_AV_GRV01_PS_PF_V5,19.567720,31.071843,19.50,32.404775,104.289840,0.0,0.0,70.25,100.0
2,GC_AV_GRV01_TL_PF_V5,55.144910,18.034325,12.50,22.805827,126.457889,0.0,0.0,24.14,100.0


# 6 - KS das variáveis

In [18]:
#Criar uma lista com o nome das variáveis sem o fpd
colunas_var_quant = list(features_quantitativas.columns.values) 

In [19]:
#Criação de um dicionário
dicionario = {}
#Aplicação de um laço para o cálculo do KS2 da lista de variáveis em colunas_var_quant
for coluna in colunas_var_quant:
   resultado = calcular_ks_2samp(df_selecao_inicial,'fpd', coluna)
   dicionario[coluna] = resultado

Um dos grupos está vazio após remover valores ausentes.


In [20]:
# 1. Converter o dicionário em DataFrame
df_ks2 = pd.DataFrame(list(dicionario.items()), columns=['VARIAVEL', 'KS2'])

# 2. Ajustar KS2: multiplicar por 100 e arredondar para 2 casas decimais
df_ks2['KS2'] = round(df_ks2['KS2'] * 100, 2)

# 3. Converter os VALORES da coluna 'VARIAVEL' para MAIÚSCULAS
df_ks2['VARIAVEL'] = df_ks2['VARIAVEL'].str.upper()  # Esta linha faz a conversão

# 4. Visualizar as 3 primeiras linhas
df_ks2.head(3)

,VARIAVEL,KS2
0,GC_AV_GRV01_IF_PF_V5,13.26
1,GC_AV_GRV01_PS_PF_V5,11.50
2,GC_AV_GRV01_TL_PF_V5,9.61


# 7 - Fisher score das variáveis

In [21]:
# 1. Calcular percentual de missings e identificar constantes
missing_percent = features_quantitativas.isna().mean() * 100
constant_cols = features_quantitativas.columns[features_quantitativas.nunique() == 1].tolist()  # Colunas com todos valores iguais

# 2. Inicializar array de Fisher Scores com zeros
fisher_scores = np.zeros(len(features_quantitativas.columns))

# 3. Identificar colunas para cálculo (não constantes e missing < 100%)
cols_validas = [
    col for col in features_quantitativas.columns 
    if (missing_percent[col] < 100) and (col not in constant_cols)
]

if cols_validas:
    # Remover linhas com NaN e alinhar target
    features_validas = features_quantitativas[cols_validas].dropna()
    y_valid = y.loc[features_validas.index]
    
    # Calcular Fisher Score e substituir NaN por 0
    f_scores, _ = f_classif(features_validas, y_valid)
    f_scores = np.nan_to_num(f_scores, nan=0.0)  # Garante NaN → 0
    
    # Atribuir valores às posições corretas
    idx_validas = [features_quantitativas.columns.get_loc(col) for col in cols_validas]
    fisher_scores[idx_validas] = f_scores

# 4. Criar DataFrame final
df_fisher = pd.DataFrame({
    'VARIAVEL': features_quantitativas.columns.str.upper(),
    'FISHER_SCORE': fisher_scores
}).sort_values('FISHER_SCORE', ascending=False)

print("\nRESULTADO DO FISHER SCORE (SCORE=0 PARA MISSING/CONSTANTES):")
display(df_fisher.head(10).style.format({'FISHER_SCORE': '{:.2f}'}))


RESULTADO DO FISHER SCORE (SCORE=0 PARA MISSING/CONSTANTES):


,VARIAVEL,FISHER_SCORE
47,PP_AV_GRV01_TL_PF_V5,10.04
133,RM_PR_CNC01_IF_PF_V5,5.53
134,RM_PR_CNC01_PS_PF_V5,5.41
138,RM_PR_CNC02_TL_PF_V5,5.34
45,PP_AV_GRV01_IF_PF_V5,5.15
145,RM_PR_ETB02_TL_PF_V5,4.14
35,PF_PC_VLR01_IF_PF_V5,4.08
93,PP_HA_VLR01_IF_PF_V5,4.04
64,PP_AV_MRC01_PS_PF_V5,4.02
44,PP_AV_AMP03_TL_PF_V5,3.96


# 8 - Mutual Information e percentual de missing das variáveis

In [22]:
def calcular_mi_por_coluna(df_features, df_target):
    """
    Calcula o Mutual Information para cada coluna, ignorando valores nulos.
    """
    resultados = []

    # Itera por todas as colunas
    for col in df_features.columns:
        resultado_col = {'VARIAVEL': col}
        
        # Pega apenas os dados não-nulos da coluna e do target
        temp_df = pd.DataFrame({
            col: df_features[col],
            'target': df_target
        }).dropna()
        
        try:
            # Calcula o MI se a amostra não estiver vazia
            mi = mutual_info_classif(
                temp_df[[col]], 
                temp_df['target'],
                random_state=42
            )[0]
            resultado_col['MI_VALOR'] = mi
        except ValueError:
            # Se a amostra for vazia, o MI é 0
            resultado_col['MI_VALOR'] = 0

        resultados.append(resultado_col)

    # Cria o DataFrame final a partir da lista de resultados
    df_mi = pd.DataFrame(resultados)
    
    # Organiza e retorna o resultado
    df_mi['VARIAVEL'] = df_mi['VARIAVEL'].str.upper()
    return df_mi.sort_values('MI_VALOR', ascending=False)

# --- Exemplo de uso ---
# Suponha que features_quantitativas seja seu DataFrame de features e y seja o target
df_resultados_mi = calcular_mi_por_coluna(features_quantitativas, y)

print("\nRESULTADO DA SELEÇÃO DE VARIÁVEIS POR MUTUAL INFORMATION:")
print(df_resultados_mi.head(3).to_markdown(index=False, floatfmt=('.4f')))


RESULTADO DA SELEÇÃO DE VARIÁVEIS POR MUTUAL INFORMATION:
| VARIAVEL          |   MI_VALOR |
|:------------------|-----------:|
| RM_PR_CAR01_TL_V5 |     2.0140 |
| RM_PR_CAR04_TL_V5 |     0.9455 |
| RM_PR_CAR11_TL_V5 |     0.7389 |


# Final

In [23]:
df_indicadores = (
    df_estatisticas
    .merge (df_final_escore, on='VARIAVEL', how='left')        # Left join com df_final_escore
    .merge (df_final,on='VARIAVEL', how='left')                # Left join com df_final
    .merge(df_ks2, on='VARIAVEL', how='left')                  # Left join com df_ks2
    .merge(df_fisher, on='VARIAVEL', how='left')               # Left join com df_fisher
    .merge(df_resultados_mi, on='VARIAVEL', how='left')        # Left join com df_resultados_mi
)
#Informações do data frame final
df_indicadores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151 entries, 0 to 150
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   VARIAVEL                     151 non-null    object 
 1   PERCENTUAL_MISSING           151 non-null    float64
 2   MEDIA                        151 non-null    float64
 3   MEDIANA                      151 non-null    float64
 4   DESVIO_PADRAO                150 non-null    float64
 5   COEFICIENTE_VARIACAO         150 non-null    float64
 6   MINIMO                       151 non-null    float64
 7   P_25                         151 non-null    float64
 8   P_75                         151 non-null    float64
 9   MAXIMO                       151 non-null    float64
 10  CORRELACAO_ESCORE_INDICADOR  139 non-null    float64
 11  CORRELACAO_ALVO              139 non-null    float64
 12  KS2                          151 non-null    float64
 13  FISHER_SCORE        

In [24]:
df_indicadores.head(3)

,VARIAVEL,PERCENTUAL_MISSING,MEDIA,MEDIANA,DESVIO_PADRAO,COEFICIENTE_VARIACAO,MINIMO,P_25,P_75,MAXIMO,CORRELACAO_ESCORE_INDICADOR,CORRELACAO_ALVO,KS2,FISHER_SCORE,MI_VALOR
0,GC_AV_GRV01_IF_PF_V5,26.706065,27.360325,8.57,33.205314,121.363010,0.0,0.0,53.63,100.0,-0.460914,0.083314,13.26,1.001805,0.003634
1,GC_AV_GRV01_PS_PF_V5,19.567720,31.071843,19.50,32.404775,104.289840,0.0,0.0,70.25,100.0,-0.393326,0.071916,11.50,1.104676,0.003589
2,GC_AV_GRV01_TL_PF_V5,55.144910,18.034325,12.50,22.805827,126.457889,0.0,0.0,24.14,100.0,-0.412440,0.063265,9.61,0.520277,0.003122


In [26]:
#Exportar para CSV
df_indicadores.to_csv('eda_selecao_varaiveis_fpd.csv', header=True, index=False, sep=';')